# Pipeline A — Step 1: Annual Panel Aggregation

**Filosofi**: Data tahunan (x3-x10) harus dimodelkan secara tahunan. Tidak membuat ilusi variasi bulanan.

**Transformasi:**
- Target `twp90_pct` → rata-rata tahunan per provinsi
- Fitur bulanan (BI rate, inflasi) → rata-rata tahunan per provinsi
- Fitur tahunan → diambil langsung (first/unique per tahun)
- `x4_tpt_pct` → rata-rata Feb + Agt per provinsi per tahun
- Log-transform pada PDRB dan tabungan

**Input:** `1_data_gathering/output/1_raw_panel_data.csv`

**Output:** `pipeline_A/output/A1_annual_panel.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '1_data_gathering' / 'output' / '1_raw_panel_data.csv').exists():
            return p
    raise FileNotFoundError('Could not find 1_raw_panel_data.csv')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / '1_data_gathering' / 'output' / '1_raw_panel_data.csv'
output_dir = ROOT / 'pipeline_A' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'A1_annual_panel.csv'

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
print(f'Loaded: {input_path}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Years: {sorted(df["tahun"].unique())}')
print(f'Provinces: {df["provinsi_id"].nunique()}')

In [ ]:
# === Agregasi ke level tahunan ===
df_annual = df.groupby(['provinsi_id', 'nama_provinsi', 'tahun']).agg(
    twp90_avg       = ('twp90_pct', 'mean'),
    twp90_std       = ('twp90_pct', 'std'),
    twp90_max       = ('twp90_pct', 'max'),
    twp90_min       = ('twp90_pct', 'min'),
    x1_bi_rate_avg  = ('x1_bi_rate_pct', 'mean'),
    x2_inflasi_avg  = ('x2_inflasi_yoy', 'mean'),
    x2_inflasi_std  = ('x2_inflasi_yoy', 'std'),
    x3_pdrb         = ('x3_pdrb_per_kapita', 'first'),
    x4_tpt          = ('x4_tpt_pct', 'mean'),          # mean of Feb+Aug values
    x5_internet     = ('x5_penetrasi_internet_pct', 'first'),
    x6_tabungan     = ('x6_tabungan_miliar', 'first'),
    x7_kc_bank      = ('x7_jumlah_kc_bank', 'first'),
    x8_ldr          = ('x8_ldr_pct', 'first'),
    x9_npl          = ('x9_npl_ratio', 'first'),
    x10_umkm        = ('x10_rasio_umkm', 'first'),
).reset_index()

print(f'Annual panel shape: {df_annual.shape[0]} rows x {df_annual.shape[1]} cols')
print(f'Expected: {df["provinsi_id"].nunique()} provinces x {df["tahun"].nunique()} years = {df["provinsi_id"].nunique() * df["tahun"].nunique()}')
display(df_annual.head())

In [ ]:
# === Log-transform variabel dengan skala besar ===
df_annual['log_pdrb'] = np.log1p(df_annual['x3_pdrb'])
df_annual['log_tabungan'] = np.log1p(df_annual['x6_tabungan'])
df_annual['log_kc_bank'] = np.log1p(df_annual['x7_kc_bank'])

# === Lag tahunan (t-1) pada fitur kunci ===
lag_features = ['twp90_avg', 'x1_bi_rate_avg', 'x2_inflasi_avg',
                'x9_npl', 'x8_ldr', 'log_pdrb']
for feat in lag_features:
    df_annual[f'{feat}_lag1'] = df_annual.groupby('provinsi_id')[feat].shift(1)

# === Delta features (year-over-year change) ===
df_annual['delta_twp90'] = df_annual.groupby('provinsi_id')['twp90_avg'].diff()
df_annual['delta_npl'] = df_annual.groupby('provinsi_id')['x9_npl'].diff()
df_annual['delta_pdrb_pct'] = df_annual.groupby('provinsi_id')['x3_pdrb'].pct_change()

# === Missing summary ===
print('\n=== Missing Values ===')
missing = df_annual.isnull().sum()
display(missing[missing > 0])

In [ ]:
# === Descriptive Statistics ===
print('=== Descriptive Statistics ===')
display(df_annual.describe().round(4))

# === Save ===
df_annual.to_csv(output_path, index=False)
print(f'\nSaved: {output_path}')
print(f'Columns: {list(df_annual.columns)}')